In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv('../01_Data/raw/household_power_consumption.csv', na_values='?')


In [ ]:
df.head()


## Making datetime column

In [ ]:
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True, errors='coerce')


In [ ]:
df[['Date','Time','datetime']].head()


In [ ]:
df = df.sort_values("datetime")


In [ ]:
df = df.set_index("datetime")


## Checking time gaps

In [ ]:
gap = df.index.to_series().diff().dt.total_seconds() / 60


In [ ]:
gap.describe()


In [ ]:
print("Normal 1 minute rows %", round((gap.dropna() == 1).mean() * 100, 2))


In [ ]:
print("Largest gap in minutes", int(gap.max()))


## Making numeric columns

In [ ]:
num_cols = ['Global_active_power','Global_reactive_power','Voltage','Global_intensity','Sub_metering_1','Sub_metering_2','Sub_metering_3']


In [ ]:
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


In [ ]:
df[num_cols].isna().sum()


## Daily data

In [ ]:
daily = df["Global_active_power"].resample("D").sum() / 60
daily = daily.to_frame("daily_kwh")


In [ ]:
daily["avg_voltage"] = df["Voltage"].resample("D").mean()


In [ ]:
daily["avg_intensity"] = df["Global_intensity"].resample("D").mean()


In [ ]:
daily["avg_reactive_power"] = df["Global_reactive_power"].resample("D").mean()


In [ ]:
daily.head()


In [ ]:
daily["records"] = df["Global_active_power"].resample("D").count()


In [ ]:
daily["coverage_pct"] = daily["records"] / 1440 * 100


In [ ]:
daily = daily[daily["coverage_pct"] >= 90]


In [ ]:
daily["dayofweek"] = daily.index.dayofweek


In [ ]:
daily["month"] = daily.index.month


In [ ]:
daily["is_weekend"] = (daily["dayofweek"] >= 5).astype(int)


## Hourly data

In [ ]:
hourly = df["Global_active_power"].resample("h").sum() / 60
hourly = hourly.to_frame("energy_kwh")


In [ ]:
hourly["avg_voltage"] = df["Voltage"].resample("h").mean()


In [ ]:
hourly["avg_intensity"] = df["Global_intensity"].resample("h").mean()


In [ ]:
hourly["records"] = df["Global_active_power"].resample("h").count()


In [ ]:
hourly["coverage_pct"] = hourly["records"] / 60 * 100


In [ ]:
hourly = hourly[hourly["coverage_pct"] >= 90]


In [ ]:
hourly["hour"] = hourly.index.hour


In [ ]:
hourly["dayofweek"] = hourly.index.dayofweek


In [ ]:
hourly["month"] = hourly.index.month


In [ ]:
hourly["is_weekend"] = (hourly["dayofweek"] >= 5).astype(int)


In [ ]:
hourly.head()


## Saving the data

In [ ]:
daily.to_csv('../01_Data/processed/daily_energy.csv', index_label='date')


In [ ]:
hourly.to_csv('../01_Data/processed/hourly_energy.csv', index_label='date_time')


In [ ]:
print("Daily rows", len(daily))
print("Hourly rows", len(hourly))
